In [ ]:
import numpy as np


In [ ]:
def _unit_vector(vector, name="vector"):
    """
    返回单位向量
    """

    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)

    if norm == 0:
        raise ValueError(f"{name} must not be a zero vector.")

    return vector / norm


In [ ]:
def transf_space_frame(node_i, node_j, reference_vector=None):
    """
    生成三维 space frame 单元的坐标变换矩阵

    每个节点 6 个自由度:
        [u, v, w, theta_x, theta_y, theta_z]

    单元两个节点的自由度顺序:
        [ui, vi, wi, theta_xi, theta_yi, theta_zi,
         uj, vj, wj, theta_xj, theta_yj, theta_zj]

    Parameters
    ----------
    node_i, node_j : array_like
        单元 i、j 节点的三维坐标，例如 [x, y, z]

    reference_vector : array_like or None
        用来确定局部 y 轴方向的参考向量。
        若不输入，默认优先使用全局 Y 方向；若杆件接近全局 Y 方向，则使用全局 Z 方向。

    Returns
    -------
    T : ndarray
        12x12 坐标变换矩阵，用于 k_global = T.T @ k_local @ T

    L : float
        单元长度
    """

    node_i = np.asarray(node_i, dtype=float)
    node_j = np.asarray(node_j, dtype=float)

    if node_i.shape != (3,) or node_j.shape != (3,):
        raise ValueError("node_i and node_j must be 3D coordinate vectors.")

    dx = node_j - node_i
    L = np.linalg.norm(dx)

    if L == 0:
        raise ValueError("Element length L must not be zero.")

    x_axis = dx / L

    if reference_vector is None:
        reference_vector = np.array([0.0, 1.0, 0.0])
        if abs(np.dot(x_axis, reference_vector)) > 0.99:
            reference_vector = np.array([0.0, 0.0, 1.0])
    else:
        reference_vector = np.asarray(reference_vector, dtype=float)

    if reference_vector.shape != (3,):
        raise ValueError("reference_vector must be a 3D vector.")

    y_trial = reference_vector - np.dot(reference_vector, x_axis) * x_axis

    if np.linalg.norm(y_trial) < 1e-12:
        raise ValueError("reference_vector must not be parallel to the element axis.")

    y_axis = _unit_vector(y_trial, "local y axis")
    z_axis = _unit_vector(np.cross(x_axis, y_axis), "local z axis")

    R = np.array([
        x_axis,
        y_axis,
        z_axis
    ], dtype=float)

    T_node = np.zeros((6, 6))
    T_node[0:3, 0:3] = R
    T_node[3:6, 3:6] = R

    T = np.zeros((12, 12))
    T[0:6, 0:6] = T_node
    T[6:12, 6:12] = T_node

    return T, L


In [ ]:
def stiffness_space_frame(EA, EIy, EIz, GJ, L=None, node_i=None, node_j=None, reference_vector=None):
    """
    三维 space frame 二节点单元刚度矩阵

    每个节点 6 个自由度:
        [u, v, w, theta_x, theta_y, theta_z]

    单元自由度顺序:
        [ui, vi, wi, theta_xi, theta_yi, theta_zi,
         uj, vj, wj, theta_xj, theta_yj, theta_zj]

    Parameters
    ----------
    EA : float
        轴向刚度 EA

    EIy : float
        绕局部 y 轴弯曲刚度 EI_y

    EIz : float
        绕局部 z 轴弯曲刚度 EI_z

    GJ : float
        扭转刚度 GJ

    L : float or None
        单元长度。若输入 node_i 和 node_j，可不输入 L。

    node_i, node_j : array_like or None
        若输入节点坐标，则函数返回全局坐标刚度矩阵；
        若只输入 L，则函数返回局部坐标刚度矩阵。

    reference_vector : array_like or None
        用于确定局部 y、z 轴方向的参考向量

    Returns
    -------
    k : ndarray
        12x12 单元刚度矩阵
    """

    T = None

    if node_i is not None and node_j is not None:
        T, L_from_nodes = transf_space_frame(node_i, node_j, reference_vector=reference_vector)
        if L is None:
            L = L_from_nodes
    elif L is None:
        raise ValueError("Input either L or both node_i and node_j.")

    if L <= 0:
        raise ValueError("L must be positive.")

    a = EA / L
    b = 12 * EIz / L**3
    c = 6 * EIz / L**2
    d = 12 * EIy / L**3
    e = 6 * EIy / L**2
    f = GJ / L
    g = 4 * EIy / L
    h = 2 * EIy / L
    m = 4 * EIz / L
    n = 2 * EIz / L

    k_local = np.array([
        [ a, 0, 0, 0, 0, 0, -a, 0, 0, 0, 0, 0],
        [ 0, b, 0, 0, 0, c, 0, -b, 0, 0, 0, c],
        [ 0, 0, d, 0, -e, 0, 0, 0, -d, 0, -e, 0],
        [ 0, 0, 0, f, 0, 0, 0, 0, 0, -f, 0, 0],
        [ 0, 0, -e, 0, g, 0, 0, 0, e, 0, h, 0],
        [ 0, c, 0, 0, 0, m, 0, -c, 0, 0, 0, n],
        [-a, 0, 0, 0, 0, 0, a, 0, 0, 0, 0, 0],
        [ 0, -b, 0, 0, 0, -c, 0, b, 0, 0, 0, -c],
        [ 0, 0, -d, 0, e, 0, 0, 0, d, 0, e, 0],
        [ 0, 0, 0, -f, 0, 0, 0, 0, 0, f, 0, 0],
        [ 0, 0, -e, 0, h, 0, 0, 0, e, 0, g, 0],
        [ 0, c, 0, 0, 0, n, 0, -c, 0, 0, 0, m]
    ], dtype=float)

    if T is None:
        return k_local

    k_global = T.T @ k_local @ T

    return k_global


In [ ]:
def assemble_K_space_frame(ke_list, connectivity, n_nodes, numbering="one"):
    """
    组装三维 space frame element 的总体刚度矩阵

    每个节点 6 个自由度:
        [u, v, w, theta_x, theta_y, theta_z]

    每个单元刚度矩阵自由度顺序:
        [ui, vi, wi, theta_xi, theta_yi, theta_zi,
         uj, vj, wj, theta_xj, theta_yj, theta_zj]
    """

    if len(ke_list) != len(connectivity):
        raise ValueError("ke_list and connectivity must have the same length.")

    dof_per_node = 6
    total_dof = dof_per_node * n_nodes

    K = np.zeros((total_dof, total_dof))

    for ke, conn in zip(ke_list, connectivity):

        ke = np.asarray(ke, dtype=float)

        if ke.shape != (12, 12):
            raise ValueError("Each space frame element stiffness matrix must be 12x12.")

        ni, nj = conn

        if numbering == "one":
            ni -= 1
            nj -= 1
        elif numbering == "zero":
            pass
        else:
            raise ValueError("numbering must be 'one' or 'zero'.")

        dofs = [
            6 * ni,      # u_i
            6 * ni + 1,  # v_i
            6 * ni + 2,  # w_i
            6 * ni + 3,  # theta_xi
            6 * ni + 4,  # theta_yi
            6 * ni + 5,  # theta_zi
            6 * nj,      # u_j
            6 * nj + 1,  # v_j
            6 * nj + 2,  # w_j
            6 * nj + 3,  # theta_xj
            6 * nj + 4,  # theta_yj
            6 * nj + 5   # theta_zj
        ]

        K[np.ix_(dofs, dofs)] += ke

    return K


In [ ]:
def dof_space_frame(node, direction, numbering="one"):
    """
    三维 space frame element 的自由度编号函数

    每个节点 6 个自由度:
        u        : x 方向位移
        v        : y 方向位移
        w        : z 方向位移
        theta_x  : 绕 x 轴转角
        theta_y  : 绕 y 轴转角
        theta_z  : 绕 z 轴转角
    """

    if numbering == "one":
        node -= 1
    elif numbering == "zero":
        pass
    else:
        raise ValueError("numbering must be 'one' or 'zero'.")

    direction = direction.lower()

    if direction in ["u", "x"]:
        return 6 * node
    elif direction in ["v", "y"]:
        return 6 * node + 1
    elif direction in ["w", "z"]:
        return 6 * node + 2
    elif direction in ["theta_x", "thetax", "rx", "rotation_x"]:
        return 6 * node + 3
    elif direction in ["theta_y", "thetay", "ry", "rotation_y"]:
        return 6 * node + 4
    elif direction in ["theta_z", "thetaz", "rz", "rotation_z"]:
        return 6 * node + 5
    else:
        raise ValueError("direction must be 'u', 'v', 'w', 'theta_x', 'theta_y', or 'theta_z'.")


In [ ]:
def solve_structure(K, force_bc=None, disp_bc=None):
    """
    求解总体刚度方程 K d = F
    """

    K = np.asarray(K, dtype=float)
    n_dof = K.shape[0]

    if K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square matrix.")

    F = np.zeros(n_dof)

    if force_bc is not None:
        for index, value in force_bc.items():
            F[index] = value

    d = np.zeros(n_dof)

    if disp_bc is None:
        disp_bc = {}

    fixed_dofs = np.array(sorted(disp_bc.keys()), dtype=int)

    for index, value in disp_bc.items():
        d[index] = value

    all_dofs = np.arange(n_dof)
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fc = K[np.ix_(free_dofs, fixed_dofs)]

    F_f = F[free_dofs]
    d_c = d[fixed_dofs]

    rhs = F_f - K_fc @ d_c
    d_f = np.linalg.solve(K_ff, rhs)

    d[free_dofs] = d_f

    internal_force = K @ d
    reaction = internal_force - F

    return d, reaction, free_dofs, fixed_dofs
